# 01 · Prepare data (run once)

Downloads FFHQ-256 (70k aligned faces, Kaggle) and COCO val2017 (5k scenes),
normalises everything to 256 px, detects 5-point face landmarks, fixes the
held-out eval split (300 FFHQ + 100 COCO, seed 42), and stores the result as
one tar on Drive. Later notebooks only unpack that tar.

Needs `kaggle.json` (kaggle.com → Settings → API → Create New Token), uploaded
when the cell asks. About 30–45 minutes on a T4 runtime. Run the cells in order.

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=False, gpu=True, requirements=('requirements.txt', 'requirements-data.txt'))
# -----------------------------------------------------------------------------

from pathlib import Path
from google.colab import files
if not Path("/root/.kaggle/kaggle.json").exists():
    up = files.upload()                      # choose kaggle.json
    Path("/root/.kaggle").mkdir(exist_ok=True)
    Path("/root/.kaggle/kaggle.json").write_bytes(up["kaggle.json"])
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

In [ ]:
!mkdir -p /content/raw/ffhq /content/raw/coco
!kaggle datasets download -d xhlulu/flickrfaceshq-dataset-nvidia-resized-256px -p /content/raw/ffhq --unzip -q
!wget -q http://images.cocodataset.org/zips/val2017.zip -O /content/raw/coco/val2017.zip
!unzip -q /content/raw/coco/val2017.zip -d /content/raw/coco && rm /content/raw/coco/val2017.zip
!find /content/raw/ffhq -type f | wc -l && find /content/raw/coco -type f | wc -l

In [ ]:
!python scripts/prepare_data.py --ffhq /content/raw/ffhq --coco /content/raw/coco/val2017 --out /content/data/prepared

In [ ]:
!mkdir -p {DRIVE_ROOT}/data
!tar -cf {DRIVE_ROOT}/data/prepared.tar -C /content/data prepared
!ls -lh {DRIVE_ROOT}/data